**`ingest_admin`**

Script examples to import administrative subdivisions for new countries

In [ ]:
from openplaces.api import get_admin1, get_admin2, get_admin3
from openplaces.core.schema import AdminId
from openplaces.io.ingest import get_recipe_data, ingest_recipe_data
from openplaces.recipe import get_recipe
from openplaces.timing import get_timer
from openplaces.utils import pretty_print

In [ ]:
REDO = False

In [ ]:
timer = get_timer('ingest_admin', verbose=True)

# Ingest data

In [ ]:
admin_id = AdminId('US')

## ``admin1``: states / departments

In [ ]:
admin1_recipe = get_recipe(admin_id, 'admin-nhgis-2020', filename='admin1')
pretty_print(admin1_recipe)

In [ ]:
ingest_recipe_data(admin1_recipe, timer=timer, redo=True)

In [ ]:
admin1 = get_admin1(recipe=admin1_recipe)
admin1.head()

## ``admin2``: counties / municipalities

In [ ]:
admin2_recipe = get_recipe(admin_id, 'admin-nhgis-2020', filename='admin2')
pretty_print(admin2_recipe)

In [ ]:
ingest_recipe_data(admin2_recipe, timer=timer, redo=True)

In [ ]:
admin2 = get_admin2(recipe=admin2_recipe, geom=True)

In [ ]:
admin2.sample(5).sort_index()

## ``admin3``: towns / county subdivisions

In [ ]:
admin3_recipe = get_recipe(admin_id, 'admin-nhgis-2020', filename='admin3')
admin3_recipe

In [ ]:
ingest_recipe_data(admin3_recipe, timer=timer, redo=True)

In [ ]:
admin3_local = get_admin3(recipe=admin3_recipe, all_columns=True)

admin3_local.head()

In [ ]:
admin3_local = get_admin3(recipe=admin3_recipe, all_columns=True)

# Create county FIPS (admin2_id_admin0)
admin3_local['admin2_id_admin0'] = admin3_local['admin3_id_admin0'].str.slice(0, 5)

# Join admin2_ids on county FIPS (admin2_id_admin0)
admin2_recipe = get_recipe('US', 'admin-nhgis-2020', filename='admin2')
admin2_crosswalk = (
    get_admin2(recipe=admin2_recipe, columns=['admin2_id_admin0'])
    .reset_index()
    .set_index('admin2_id_admin0')['admin2_id']
)
admin3_local = admin3_local.join(admin2_crosswalk, on='admin2_id_admin0')
admin3_local

Test code from Claude - seems to do what is needed

In [ ]:
"""
Generate unique two-letter admin ID codes for administrative units

Level-agnostic design: works for any parent-child administrative relationship
(e.g., admin1->admin2, admin2->admin3, county->town, state->county, etc.)

Strategy:
1. Initials from multi-word/hyphenated names (e.g., "Los Angeles" → "LA")
2. First two letters of name (for single-word names)
3. Any two letters from name
4. Use 'C' or 'T' suffix for cities/townships when helpful
5. Swap existing codes to free up better matches
6. Three-letter codes if two-letter space exhausted
7. Sequential numbering as last resort
"""

from itertools import combinations

import numpy as np
import pandas as pd

from openplaces.core.constants import STRING_SEPARATOR_WITHIN_IDS


def generate_admin_ids(
    df,
    new_admin_id_col='admin3_id',
    parent_admin_id_col='admin2_id',
    name_col='name',
    name_long_col=None,
    id_separator=STRING_SEPARATOR_WITHIN_IDS,
):
    """
    Generate unique two-letter administrative unit codes within parent units.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe with administrative unit data
    new_admin_id_col : str
        Name for the new administrative ID column (default 'admin3_id')
    parent_admin_id_col : str
        Column name containing parent admin ID (e.g., 'admin2_id')
    name_col : str
        Column name containing subdivision name
    name_long_col : str, optional
        Column name containing long-form name with suffixes (e.g., 'city', 'township')
        If None or column doesn't exist, city/township detection is skipped
    id_separator : str
        Separator to use in IDs (default '_')

    Returns
    -------
    pd.DataFrame
        DataFrame indexed by new_admin_id_col with diagnostics column

    Raises
    ------
    ValueError
        If unable to generate unique IDs for all rows
    """

    # Work on a copy
    admin = df.copy()

    # Auto-generate source column name
    id_source_col = new_admin_id_col + '_source'

    # Initialize columns
    admin[new_admin_id_col] = None
    admin[id_source_col] = None
    # Ensure _name_clean never has None/NaN - use empty string as fallback
    admin['_name_clean'] = (
        admin[name_col]
        .fillna('')
        .str.upper()
        .str.replace(' ', '', regex=False)
        .str.replace('-', '', regex=False)
    )

    # Detect city/township/borough types from name_long (if available)
    use_name_long = name_long_col is not None and name_long_col in admin.columns
    if use_name_long:
        admin['_is_city'] = admin[name_long_col].str.contains(
            ' city$', case=False, na=False
        )
        admin['_is_township'] = admin[name_long_col].str.contains(
            ' township$', case=False, na=False
        )
        admin['_is_borough'] = admin[name_long_col].str.contains(
            ' borough$', case=False, na=False
        )
    else:
        admin['_is_city'] = False
        admin['_is_township'] = False
        admin['_is_borough'] = False

    # Sort for consistent processing
    admin = admin.sort_values([parent_admin_id_col, name_col]).copy()

    # Track used IDs globally
    used_ids = set()

    # Priority 1: First letter + first letter of second word (hyphenated/multi-word)
    print("Priority 1: Initials from multi-word/hyphenated names...")
    mask = admin[new_admin_id_col].isna()
    if mask.any():
        # Extract initials from multi-word names (treat hyphens as word separators)
        names = admin.loc[mask, name_col].str.upper().str.replace('-', ' ')
        has_multiple_words = names.str.contains(' ', na=False)
        if has_multiple_words.any():
            words_split = names[has_multiple_words].str.split(' ', n=1)
            codes = words_split.str[0].str[0] + words_split.str[1].str[0]
            candidate_ids = (
                admin.loc[mask & has_multiple_words, parent_admin_id_col]
                + id_separator
                + codes
            )
            is_unique = ~candidate_ids.duplicated(keep=False) & ~candidate_ids.isin(
                used_ids
            )
            idx_to_update = mask & has_multiple_words
            admin.loc[idx_to_update & is_unique, new_admin_id_col] = candidate_ids[
                is_unique
            ]
            admin.loc[idx_to_update & is_unique, id_source_col] = 'initials'
            used_ids.update(candidate_ids[is_unique])

    print(f"  Assigned: {admin[new_admin_id_col].notna().sum()}/{len(admin)}")

    # Priority 2: First two letters (for single-word names)
    print("Priority 2: First two letters...")
    mask = admin[new_admin_id_col].isna() & (admin['_name_clean'].str.len() >= 2)
    if mask.any():
        codes = admin.loc[mask, '_name_clean'].str[:2]
        candidate_ids = admin.loc[mask, parent_admin_id_col] + id_separator + codes
        # Only assign IDs that are unique within this batch and not already used
        is_unique = ~candidate_ids.duplicated(keep=False) & ~candidate_ids.isin(
            used_ids
        )
        admin.loc[mask & is_unique, new_admin_id_col] = candidate_ids[is_unique]
        admin.loc[mask & is_unique, id_source_col] = 'first2'
        used_ids.update(candidate_ids[is_unique])

    print(f"  Assigned: {admin[new_admin_id_col].notna().sum()}/{len(admin)}")

    # Priority 3: Any two letters from name
    print("Priority 3: Any two letters from name...")
    mask = admin[new_admin_id_col].isna()
    unassigned = admin[mask].copy()

    if len(unassigned) > 0:
        # Pre-compute all needed data
        indices = unassigned.index.tolist()
        names_clean = unassigned['_name_clean'].tolist()
        parent_ids = unassigned[parent_admin_id_col].tolist()

        # Process in batch
        for i, (idx, name_clean, parent_id) in enumerate(
            zip(indices, names_clean, parent_ids)
        ):
            if len(name_clean) < 2:
                continue

            # Try combinations
            for c1, c2 in combinations(name_clean, 2):
                code = c1 + c2
                new_id = parent_id + id_separator + code
                if new_id not in used_ids:
                    admin.loc[idx, new_admin_id_col] = new_id
                    admin.loc[idx, id_source_col] = 'any2'
                    used_ids.add(new_id)
                    break

    print(f"  Assigned: {admin[new_admin_id_col].notna().sum()}/{len(admin)}")

    # Priority 3b: Letter + number combinations (for names with few letters)
    print("Priority 3b: Letter + number combinations...")
    mask = admin[new_admin_id_col].isna()
    unassigned = admin[mask].copy()

    if len(unassigned) > 0:
        # Pre-extract all needed data in batch
        indices = unassigned.index.tolist()
        parent_ids = unassigned[parent_admin_id_col].tolist()

        # Extract letters and numbers from name and name_long columns
        names_upper = unassigned[name_col].fillna('').str.upper().tolist()
        if use_name_long:
            names_long_upper = unassigned[name_long_col].fillna('').str.upper().tolist()
        else:
            names_long_upper = [''] * len(names_upper)

        # Process in batch
        for idx, parent_id, name_upper, name_long_upper in zip(
            indices, parent_ids, names_upper, names_long_upper
        ):
            combined_name = name_upper + ' ' + name_long_upper
            letters = [c for c in combined_name if c.isalpha()]
            numbers = [c for c in combined_name if c.isdigit()]

            # Try letter + number combinations
            found = False
            if letters and numbers:
                for letter in letters:
                    for number in numbers:
                        code = letter + number
                        new_id = parent_id + id_separator + code
                        if new_id not in used_ids:
                            admin.loc[idx, new_admin_id_col] = new_id
                            admin.loc[idx, id_source_col] = 'letter_num'
                            used_ids.add(new_id)
                            found = True
                            break
                    if found:
                        break

            # If no letters at all, use X + number
            if not found and not letters and numbers:
                for number in numbers:
                    code = 'X' + number
                    new_id = parent_id + id_separator + code
                    if new_id not in used_ids:
                        admin.loc[idx, new_admin_id_col] = new_id
                        admin.loc[idx, id_source_col] = 'x_num'
                        used_ids.add(new_id)
                        break

    print(f"  Assigned: {admin[new_admin_id_col].notna().sum()}/{len(admin)}")

    # Priority 4: Try C/T suffix for cities/townships
    print("Priority 4: Using C/T suffix for cities/townships...")
    mask = admin[new_admin_id_col].isna() & (admin['_is_city'] | admin['_is_township'])
    if mask.any():
        first_letter = admin.loc[mask, '_name_clean'].str[0]
        suffix = admin.loc[mask, '_is_city'].map({True: 'C', False: 'T'})
        codes = first_letter + suffix
        candidate_ids = admin.loc[mask, parent_admin_id_col] + id_separator + codes
        is_unique = ~candidate_ids.duplicated(keep=False) & ~candidate_ids.isin(
            used_ids
        )
        admin.loc[mask & is_unique, new_admin_id_col] = candidate_ids[is_unique]
        admin.loc[mask & is_unique, id_source_col] = 'suffix_CT'
        used_ids.update(candidate_ids[is_unique])

    print(f"  Assigned: {admin[new_admin_id_col].notna().sum()}/{len(admin)}")

    # Priority 5: Swap existing codes to free up better matches
    print("Priority 5: Swapping existing codes...")
    mask = admin[new_admin_id_col].isna()
    swaps_made = 0
    unassigned = admin[mask].copy()

    if len(unassigned) > 0:
        indices = unassigned.index.tolist()
        names_clean = unassigned['_name_clean'].tolist()
        parent_ids = unassigned[parent_admin_id_col].tolist()

        for idx, name_clean, parent_id in zip(indices, names_clean, parent_ids):
            if len(name_clean) < 2:
                continue

            found = False
            for c1, c2 in combinations(name_clean, 2):
                code = c1 + c2
                new_id = parent_id + id_separator + code

                # Check if this ID is already taken
                if new_id in used_ids:
                    # Find who has it
                    existing_mask = admin[new_admin_id_col] == new_id
                    if not existing_mask.any():
                        continue
                    existing_idx = existing_mask.idxmax()
                    existing_name_clean = admin.at[existing_idx, '_name_clean']
                    existing_parent_id = admin.at[existing_idx, parent_admin_id_col]

                    # Skip if existing name is too short
                    if len(existing_name_clean) < 2:
                        continue

                    # Only swap if they're in the same parent unit
                    if existing_parent_id == parent_id:
                        # Try to find alternative for existing holder
                        swap_found = False
                        for d1, d2 in combinations(existing_name_clean, 2):
                            alt_code = d1 + d2
                            alt_new_id = existing_parent_id + id_separator + alt_code
                            if alt_new_id not in used_ids and alt_code != code:
                                # Perform swap
                                used_ids.remove(new_id)
                                admin.loc[existing_idx, new_admin_id_col] = alt_new_id
                                admin.loc[existing_idx, id_source_col] = 'swapped'
                                used_ids.add(alt_new_id)

                                admin.loc[idx, new_admin_id_col] = new_id
                                admin.loc[idx, id_source_col] = 'any2_after_swap'
                                used_ids.add(new_id)

                                swap_found = True
                                swaps_made += 1
                                break

                        if swap_found:
                            found = True
                            break

            if found:
                break

    print(f"  Swaps made: {swaps_made}")
    print(f"  Assigned: {admin[new_admin_id_col].notna().sum()}/{len(admin)}")

    # Priority 6: Three-letter codes for remaining
    print("Priority 6: Three-letter codes...")
    mask = admin[new_admin_id_col].isna()

    # First try: first three letters (vectorized)
    if mask.any():
        has_three = admin.loc[mask, '_name_clean'].str.len() >= 3
        if has_three.any():
            codes = admin.loc[mask & has_three, '_name_clean'].str[:3]
            candidate_ids = (
                admin.loc[mask & has_three, parent_admin_id_col] + id_separator + codes
            )
            is_unique = ~candidate_ids.duplicated(keep=False) & ~candidate_ids.isin(
                used_ids
            )
            admin.loc[mask & has_three & is_unique, new_admin_id_col] = candidate_ids[
                is_unique
            ]
            admin.loc[mask & has_three & is_unique, id_source_col] = 'first3'
            used_ids.update(candidate_ids[is_unique])

    # Second try: any three letters (needs iterative approach)
    mask = admin[new_admin_id_col].isna()
    unassigned = admin[mask].copy()

    if len(unassigned) > 0:
        indices = unassigned.index.tolist()
        names_clean = unassigned['_name_clean'].tolist()
        parent_ids = unassigned[parent_admin_id_col].tolist()

        for idx, name_clean, parent_id in zip(indices, names_clean, parent_ids):
            if len(name_clean) < 3:
                continue

            for c1, c2, c3 in combinations(name_clean, 3):
                code = c1 + c2 + c3
                new_id = parent_id + id_separator + code
                if new_id not in used_ids:
                    admin.loc[idx, new_admin_id_col] = new_id
                    admin.loc[idx, id_source_col] = 'any3'
                    used_ids.add(new_id)
                    break

    print(f"  Assigned: {admin[new_admin_id_col].notna().sum()}/{len(admin)}")

    # Last resort: Sequential numbering
    print("Priority 7: Sequential numbering...")
    mask = admin[new_admin_id_col].isna()
    if mask.any():
        # Group by parent_admin_id_col and assign sequential numbers
        remaining = admin[mask].groupby(parent_admin_id_col)
        for parent_id, group in remaining:
            for i, idx in enumerate(group.index, start=1):
                code = f'X{i:02d}'
                new_id = parent_id + id_separator + code
                # Ensure uniqueness
                counter = 1
                while new_id in used_ids:
                    code = f'X{i:02d}{chr(64+counter)}'
                    new_id = parent_id + id_separator + code
                    counter += 1

                admin.loc[idx, new_admin_id_col] = new_id
                admin.loc[idx, id_source_col] = 'sequential'
                used_ids.add(new_id)

    print(f"  Final assigned: {admin[new_admin_id_col].notna().sum()}/{len(admin)}")

    # Verify uniqueness
    if admin[new_admin_id_col].isna().any():
        n_missing = admin[new_admin_id_col].isna().sum()
        raise ValueError(f"Failed to assign IDs to {n_missing} rows")

    if admin[new_admin_id_col].duplicated().any():
        n_dupes = admin[new_admin_id_col].duplicated().sum()
        dupes = admin[admin[new_admin_id_col].duplicated(keep=False)][
            [new_admin_id_col, name_col, parent_admin_id_col]
        ]
        raise ValueError(f"Found {n_dupes} duplicate IDs:\n{dupes}")

    print("\n✓ All IDs assigned and verified unique!")

    # Print summary statistics
    print("\nID Generation Summary:")
    print(admin[id_source_col].value_counts().to_string())

    # Clean up temporary columns and set index
    admin = admin.drop(
        columns=['_name_clean', '_is_city', '_is_township', '_is_borough']
    )
    admin = admin.set_index(new_admin_id_col)

    return admin

result = generate_admin_ids(admin3_local, 'admin3_id', 'admin2_id')
result
timer.mark('Done')
result[['name', 'name_long']].sample(25)